<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
#shakespeare dataset

--2026-06-14 19:16:10--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.04s   

2026-06-14 19:16:10 (27.1 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [2]:
with (open('input.txt', 'r', encoding='utf-8')) as f:
  text = f.read()

In [3]:
print(f"Length of data set in characters: {len(text)}")

Length of data set in characters: 1115394


In [4]:
print(f"First 1000 characters of data set: \n{text[:1000]}")

First 1000 characters of data set: 
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for

In [5]:
characters =sorted(list(set(text))) #set() creates a collection of unique elements then we make a list and then sorted it
print(characters)

['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


In [6]:
vocab_size = len(characters)
print(f"Vocab size: {vocab_size}")

Vocab size: 65


In [7]:
print ( ''.join(characters))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [8]:
stoi = { ch:i for i,ch in enumerate(characters) } #stoi means string to integer basically from characters list we are picking each character one by one and assign their index or token id
itos = { i:ch for i,ch in enumerate(characters) } #itos means integer to string for decoding to string we take same characters list and here we assign index to characters so when we give index it gives character

encoder = lambda s: [stoi[ch] for ch in s] #here basically we get s as input string then we iterate through that string character by character and each character go to stoi and retrieve its index id and return list of these
decoder = lambda l : ''. join(itos[i] for i in l ) #and here we give integer list iterate it and get every index character and join it

In [9]:
print(encoder("hello world"))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]


In [10]:
print(decoder([46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]))

hello world


In [11]:
# now we encode all our dataset but also convert them into tensor
import torch
data = torch.tensor(encoder(text), dtype=torch.long) #torch.long = 64-bit integer So every value in the tensor is stored as a whole number (no decimals), taking up 64 bits of memory becuase these are lookup indices as after converting each integer into embedding these int work as index to embedding table so it cannot be float
print(data.shape, data.dtype)
print(data[:10])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47])


In [12]:
n = int(0.9 * len(data)) #90% of training data
train_data = data[:n]
validation_data = data[n:] #test data

In [13]:
block_size = 8 #means in each batch we take 8 characters in order
train_data[:block_size+1] #+1 beacuse for input in train data is 0 or any index to block size but for output since we do character by character prediction so 0 index input ans is 1 so till block size input ans is block size +1

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [14]:
x = train_data[:block_size] #input
y = train_data[1:block_size+1] #output
for t in range(block_size):
  context = x[:t+1] # as we are taking that list now one by one so x[0: till t+1] if 3rd value give as input so t=2 as 0,1,2 so we take x[0:t+1 = 3] which means 18,47,56 from above printed list
  target=y[t] # as y already y start from one index after[1:block_size +1] so if t=2 y[2] is 57 from above list
  print(f"When input is {context} target is {target}")

When input is tensor([18]) target is 47
When input is tensor([18, 47]) target is 56
When input is tensor([18, 47, 56]) target is 57
When input is tensor([18, 47, 56, 57]) target is 58
When input is tensor([18, 47, 56, 57, 58]) target is 1
When input is tensor([18, 47, 56, 57, 58,  1]) target is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]) target is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]) target is 58


In [15]:
torch.manual_seed(1337) #means it fix torch.rand  to pick from same fix position in every time this file run so it helps in debugging
batch_size = 4 # 4 sequence at a time go for training . how many independent sequences will we process in parallel?
block_size = 8 # each sequence has 8 characters in it. what is the maximum context length for predictions?

def get_batch(split):
  data = train_data if split == "train" else validation_data
  indices = torch.randint(len(data) - block_size , (batch_size,)) # torch.randint(low, high, size) means first one is(lower limit, higher limit, size)
  #lower limit means from selection of numbers(in our case index) we can select as low as define in limit suppose we have 1 to 10 numbers and if we give lower_limit 2 means we can select from 2 to 10 only
  #same as upper limit as what max number or index we can select from sequence like if upper limit is 8 lower is 2 from 1 to 10 so we can select only number 2 to 8 and torch randomly select any number
  #so in our case we have 2 arguments only means lower limit is 0 so we can select from 0 and upper limit is len(data) 1115394 - block size which means from 0 to 115386 it select any number and we are givimg batch size is 4 means it select any 4 number from this range
  #and these 4 number act as index we pass those index to our data and get characters(their token id) so if it consider like 55 so it goes to data and get token id of character at 55 index
  #then we have to take in block size for input and output so we take as 55 : 55+ block size means 55 : 63 position characters for input
  #now one thing in upper limit why we are - block size beacuse we take full length of data and it select number 1115392 so if we give that index to data then we have only one charcter left after that as for 1115394 len we have index 0 to 1115393 so for 1115392 we have only 2 characters which doesn't match our block size and causes problem so our last starting point of sequence index is 1115386 so from this to 1115893 we get one batch of 8 charcters
  x = torch.stack([data[ i : i+block_size] for i in indices]) #selecting one number from indices as we have as any number as batch size so for each batch we get input and output
  y = torch.stack([data[ i + 1: i+ block_size+ 1] for i in indices])
  #torch.stach basically takes all batches and put it in a new tensor[[],[],[],[]] so now x(4batch size, 8) means one tensor conatin 4 tensor each have 8 numbers in it
  return x, y

x_batch, y_batch = get_batch("train")
print(f"Input shape and dtype: {x_batch.shape}, {x_batch.dtype}")
print(f"Output shape and dtype: {y_batch.shape}, {y_batch.dtype}")
print(x_batch)
print(y_batch)

for b in range(batch_size):
  for t in range(block_size):
    context = x_batch[b, :t+1] #[b,] means if batch is 0 so we take tensor's first tensor and its elemenet 0 till t+1
    target = y_batch[b, t] #same here [b,] from b=0 take y [t]
    print(f"When input is {context.tolist()} target is {target}") #here context.tolist only for readability like tensor print as tensor.... so we give tolist so it just direct print list we donot do it with target as it is singke number only


Input shape and dtype: torch.Size([4, 8]), torch.int64
Output shape and dtype: torch.Size([4, 8]), torch.int64
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
When input is [24] target is 43
When input is [24, 43] target is 58
When input is [24, 43, 58] target is 5
When input is [24, 43, 58, 5] target is 57
When input is [24, 43, 58, 5, 57] target is 1
When input is [24, 43, 58, 5, 57, 1] target is 46
When input is [24, 43, 58, 5, 57, 1, 46] target is 43
When input is [24, 43, 58, 5, 57, 1, 46, 43] target is 39
When input is [44] target is 53
When input is [44, 53] target is 56
When input is [44, 53, 56] target is 1
When input is [44, 53, 56, 1] target is 58
When input is [44, 53, 56, 1, 58] target is

In [16]:
print(x_batch) #our input to transformer

tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])


In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
  def __init__(self, vocab_size):
    super().__init__() #first class its super class nn.module init function then this one
    self.token_embedding_table = nn.Embedding(vocab_size, vocab_size) #we form a lookup table for each vocab which represnted w.r.t to all other vocab conatin score how they are related first it is random then we train it
  def forward(self, idx, targets=None):
    logits = self.token_embedding_table(idx) # (B,T,C) means now we take our each batch and select each number represnt to letter and now define each letter into 65 dimension as per dimension table(llokup table)
    if targets is None:
      loss = None
    else:
      B, T, C = logits.shape
      logits = logits.view(B*T, C) # as cross entropy take logits in (B*T, C) and target as in one dimension(B*T)
      targets = targets.view(B*T)
      loss = F.cross_entropy(logits, targets)
    return logits, loss
  def generate(self, idx, max_new_tokens):
    for _ in range(max_new_tokens):
      logits, loss = self(idx) #nn.module has a apecial method __call__ that runs actual function when we call the object self(idx) where self is model like a function
      logits = logits[:,-1,:] # take only last token from each batch with all its dimension
      # for probabilities we apply softmax
      probs = F.softmax(logits, dim= -1)
      idx_next = torch.multinomial(probs, num_samples = 1) #so now from each batch we only have last token which dimension we convert into probability now multinomial pick one value randomly from dimension mostly high but can select other one that dimesnion show our last token more relted to all 65 one so we pick one that would be our next token ans that is how we learn relationship and we done it for all batches
      idx = torch.cat((idx, idx_next), dim=1) # (B, T+1) idx is our input that we expand it on dimesniona and idx next is predicted one so we keep conactenating it each to get better result

    return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(x_batch, y_batch)
print(logits.shape)
print(loss)

print(decoder(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist())) #we give some random input by torch.zeroes(1,1) and then it give some token value as generate run till 100 so it produce 100 token then we decode it into letters


torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [25]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [26]:
batch_size = 32
for steps in range(10000):
  xb, yb = get_batch("train")
  logits, loss = m(xb, yb)
  optimizer.zero_grad(set_to_none=True)
  loss.backward()
  optimizer.step()

print(loss.item())

2.382369041442871


In [27]:
print(decoder(m.generate(idx = torch.zeros((1,1), dtype=torch.long), max_new_tokens=500)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulseecherd d o blllando;LUCEO, oraingofof win!
RIfans picspeserer hee tha,
TOFonk? me ain ckntoty ded. bo'llll st ta d:
ELIS me hurf lal y, ma dus pe athouo
BEY:! Indy; by s afreanoo adicererupa anse tecorro llaus a!
OLeneerithesinthengove fal amas trr
TI ar I t, mes, n IUSt my w, fredeeyove
THek' merer, dd
We ntem lud engitheso; cer ize helorowaginte the?
Thak orblyoruldvicee chot, p,
Bealivolde Th li


In [28]:
torch.manual_seed(42)
a = torch.tril(torch.ones(3, 3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b
print('a=')
print(a)
print('--')
print('b=')
print(b)
print('--')
print('c=')
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
--
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
--
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


In [29]:
torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time, channels
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [30]:
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [31]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x # (B, T, T) @ (B, T, C) ----> (B, T, C)
torch.allclose(xbow, xbow2)

False

In [32]:
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)

False

In [33]:
torch.manual_seed(1337)
B,T,C = 4,8,32 # batch, time, channels
x = torch.randn(B,T,C)

# let's see a single Head perform self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)   # (B, T, 16)
q = query(x) # (B, T, 16)
wei =  q @ k.transpose(-2, -1) # (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v
#out = wei @ x

out.shape

torch.Size([4, 8, 16])

In [34]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [35]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2, -1) * head_size**-0.5

In [36]:
k.var()

tensor(1.0449)

In [37]:
q.var()

tensor(1.0700)

In [38]:
wei.var()

tensor(1.0918)

In [39]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5]), dim=-1)

tensor([0.1925, 0.1426, 0.2351, 0.1426, 0.2872])

In [40]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.5])*8, dim=-1) # gets too peaky, converges to one-hot

tensor([0.0326, 0.0030, 0.1615, 0.0030, 0.8000])

In [41]:
class LayerNorm1d: # (used to be BatchNorm1d)

  def __init__(self, dim, eps=1e-5, momentum=0.1):
    self.eps = eps
    self.gamma = torch.ones(dim)
    self.beta = torch.zeros(dim)

  def __call__(self, x):
    # calculate the forward pass
    xmean = x.mean(1, keepdim=True) # batch mean
    xvar = x.var(1, keepdim=True) # batch variance
    xhat = (x - xmean) / torch.sqrt(xvar + self.eps) # normalize to unit variance
    self.out = self.gamma * xhat + self.beta
    return self.out

  def parameters(self):
    return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm1d(100)
x = torch.randn(32, 100) # batch size 32 of 100-dimensional vectors
x = module(x)
x.shape

torch.Size([32, 100])

In [42]:
x[:,0].mean(), x[:,0].std() # mean,std of one feature across all batch inputs

(tensor(0.1469), tensor(0.8803))

In [43]:
x[0,:].mean(), x[0,:].std() # mean,std of a single input from the batch, of its features

(tensor(-9.5367e-09), tensor(1.0000))

**The idea behind this gpt is we are predicting character by character like if we give one character the model in trainig set learn what could be the next character and same in prediction it predicts one character at a time in current times mostly predicting occur token by token like we convert words into sub words and make it a token but for now we are making it simpler and doing character level predictions but since gpu are efficient in parallel processing so in training there may be n number of learning of next predicting token going on by model like if it want to laern hello so in same time using batch size  and block size learn after h e occur simultaneously it is learining after he l occur , and after hel l occur and after hell o occur so this whole can be done at t = 1(in actual one forward pass) as gpu done these matrix multiplictaion by threads simultenously **
A forward pass is just one full run of the input through the model from start to finish — input goes in, prediction comes out.
**SO like if training data is hello so we pass input as hell and model learn for x1 h y1 e for x2 e y2 l for x3 l y3 l fro x4 l y4 0 so our y is like ello in one forward pass from training data**
SO nowadays voab size can be 50000 which is of different token but since we are characters it is all characters using in file which is 65

And since we are using gpu so we give batch size as batch size is 4 means from our training data pick randomly as we are using troch.rand 4 sequence of charcters like if block size is 2 then we 4 sequence of 2 characters each and in block charcaters are in order so model learn what comes after what charcater mostly but we can do this learning in parallel in gpu so we mostly pick random batches of block size seq and model learn it
why random?
|beacuse if we go order in batches and also in these batches each character are also in order which is basically batch size then model become overfit so to avoid that we pick random batches of block size so tensor shape become (batch size, block size)